# [실습] Ollama와 LangChain을 이용한 오픈 RAG 만들기

RAG는 Retrieval-Augmented Generation (RAG) 의 약자로, 질문이 주어지면 관련 있는 문서를 찾아 프롬프트에 추가하는 방식의 어플리케이션입니다.   
RAG의 과정은 아래와 같이 진행됩니다.
1. Indexing : 문서를 받아 검색이 잘 되도록 저장합니다.
1. Processing : 입력 쿼리를 전처리하여 검색에 적절한 형태로 변환합니다<br>(여기서는 수행하지 않습니다)
1. Search(Retrieval) : 질문이 주어진 상황에서 가장 필요한 참고자료를 검색합니다.
1. Augmenting : Retrieval의 결과와 입력 프롬프트를 이용해 LLM에 전달할 프롬프트를 생성합니다.
1. Generation : LLM이 출력을 생성합니다.

## 라이브러리 설치  

랭체인 관련 라이브러리와 벡터 데이터베이스 라이브러리를 설치합니다.  
<br>

`sentence_transformers`: 트랜스포머 계열의 공개 임베딩 모델을 사용할 수 있습니다.    
`langchain_chroma`: ChromaDB를 이용해 벡터 데이터베이스를 구성합니다.    
`langchain_ollama`: Ollama에서 서빙하는 LLM을 불러옵니다.

In [1]:
# LangChain + LLM
%pip install langchain-huggingface langchain langchain-openai langchain-docling langchain-community langchain-ollama -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# RAG
%pip install docling sentence_transformers langchain_chroma ragas sacrebleu -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# 기타
%pip install dotenv jsonlines beautifulsoup4 -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain-qdrant


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
%pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 94.5 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# 시스템/유틸리티
import os
import gc
import ast
import csv
import uuid
import datetime
import re

# API & 환경 설정
import openai
from openai import AsyncOpenAI
from dotenv import load_dotenv

# 데이터 처리 및 시각화
import pandas as pd
from tqdm import tqdm
from glob import glob

# 네트워크/웹 관련
import requests
import jsonlines
import bs4

# 벡터/임베딩/LLM 관련
import torch
from sentence_transformers import SentenceTransformer
from langchain_ollama import ChatOllama
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

# LangChain 핵심 모듈
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_docling import DoclingLoader

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams


# 기타 필수
import nest_asyncio

import zipfile
import os

In [4]:
load_dotenv('.env', override=True)


client = openai.OpenAI()
# API 키 검증하기
try: client.models.list(); print("OPENAI_API_KEY가 정상적으로 설정되어 있습니다.")
except:  print(f"OpenAI API 키가 유효하지 않습니다!")

OPENAI_API_KEY가 정상적으로 설정되어 있습니다.


## LLM과 임베딩 모델 구성하기     

이번 실습에서는 GPT 계열 모델을 사용하지 않고, 오픈 모델을 불러오겠습니다.

## Ollama를 통해 LLM 불러오기


https://ollama.com/


올라마는 가장 간단하게 오픈 모델을 구동할 수 있는 방식입니다.   
추론 최적화/병렬 처리 등의 기능은 뛰어나지 않지만  
CPU에서도 구동이 가능하며, UI도 지원합니다.

### Ollama 설치하고 서빙하기    
터미널을 이용해 아래의 커맨드를 입력합니다.

- 설치

```curl -fsSL https://ollama.com/install.sh | sh```

- RAG를 위한 Context Window 늘리기 (기본 Context 4096) + 모델 Unload 시간 늘리기

```export OLLAMA_CONTEXT_LENGTH=16384 OLLAMA_KEEP_ALIVE=1200```

- 서빙 (& 으로 백그라운드 실행)

```ollama serve &```

이후 엔터를 입력해 터미널 입력창을 다시 띄웁니다.



올라마는 기본적으로 11434 포트에서 실행됩니다.   
Pull 을 통해 허깅페이스 모델의 주소나 Ollama 주소를 입력하면 모델을 불러와 저장합니다.

불러올 수 있는 모델의 목록은 https://ollama.com/search 에서 확인할 수 있습니다.   

터미널에서 아래 커맨드를 입력해 nemotron-cascade-2:30b 모델을 불러옵니다.    
```
ollama pull nemotron-cascade-2:30b
```

In [12]:
llm = ChatOllama(model='llama3:8b', # 디스크 용량이 부족해 nemotron-cascade-2:30b 대신 llama3:8b pull 받음
                 #reasoning = 'True', # llama3:8b는 옵션에 reasoning이 없어서 주석처리함
                 temperature=1.0, timeout=60)
                 # Reasoning: 추론 모델에서만 사용

test = '안녕!!!!!!!!!!!!'
llm.invoke(test)

AIMessage(content="😊 안녕하세요!!!!! 🎉 How's it going? 😄", additional_kwargs={}, response_metadata={'model': 'llama3:8b', 'created_at': '2026-03-25T12:24:56.970462088Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4643286958, 'load_duration': 4462197585, 'prompt_eval_count': 14, 'prompt_eval_duration': 16782512, 'eval_count': 16, 'eval_duration': 141091670, 'logprobs': None, 'model_name': 'llama3:8b', 'model_provider': 'ollama'}, id='lc_run--019d24f4-6f66-7bf0-a051-3e6b79917c33-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 16, 'total_tokens': 30})

In [13]:
# 스트리밍 (Thinking 제외)
test = 'ㅠㅠ'
for s in llm.stream(test):
    print(s.content, end='')

It looks like you're expressing a bit of sadness or frustration? 😔 Would you like to talk about what's on your mind and why you're feeling that way? I'm here to listen and offer support if you need it. 💕

In [11]:
# 스트리밍 (Thinking 포함) # 위에  reasoning = False 로 수정함

think = False
test = 'ㅠㅠ'
for s in llm.stream(test):
    reasoning_token = s.additional_kwargs.get('reasoning_content')

    if reasoning_token: # 생각중
        if not think:
            print('<THINK>\n', end='') # 처음에 <THINK> 출력
            think = True
        print(reasoning_token, end='')

    else: # 생각 끝
        if think:
            print('\n</THINK>') # 마지막에 </THINK> 출력
            think = False
        print(s.content, end='')

ㅠㅠ (ㅠㅠ)  
**It looks like you're feeling sad or overwhelmed, and I'm here to listen.** 🌧️  

You don’t have to explain—just know:  
💙 *Your feelings are valid.*  
💙 *You’re not alone.*  
💙 *It’s okay to take a breath.*  

Would you like to talk about what’s on your mind?  
Or maybe just need a quiet moment? I’m right here. 🫶  

*(No pressure to respond—just sharing the space with you.)*

## 데이터 불러오기   

실습에 필요한 데이터를 불러옵니다.   




In [23]:
import os
os.chdir("/workspace/lab")
print(os.getcwd())

/workspace/lab


In [27]:
import zipfile
import os

# 압축 파일 이름 (현재 위치)
zip_filename = "RAG_data.zip"

# 압축 해제 경로 (data 폴더)
extract_path = "data"

try:
    # data 폴더 없으면 생성
    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print(f"성공: {zip_filename}의 압축을 {extract_path}/ 폴더에 풀었습니다.")

except FileNotFoundError:
    print(f"오류: {zip_filename} 파일을 찾을 수 없습니다.")

except zipfile.BadZipFile:
    print("오류: 손상되었거나 유효하지 않은 zip 파일입니다.")

성공: RAG_data.zip의 압축을 data/ 폴더에 풀었습니다.


In [33]:
import shutil

inner_folder = os.path.join(extract_path, "RAG_data")

if os.path.exists(inner_folder):
    for item in os.listdir(inner_folder):
        src = os.path.join(inner_folder, item)
        dst = os.path.join(extract_path, item)
        shutil.move(src, dst)

    shutil.rmtree(inner_folder)

In [37]:
docs = glob('data/pdfs/*.pdf')
docs

['data/pdfs/여신일반.pdf', 'data/pdfs/수신일반.pdf', 'data/pdfs/금융실명거래 업무해설.pdf']

다양한 문서의 타입을 고려하여, 적절한 파서를 선택해야 합니다.

- PyMuPDF: 가장 기본적인 파서입니다. 텍스트 요소만을 추출하여 나열하며, 테이블 등의 요소에 약합니다.
- Docling: IBM의 오픈소스 파서로, 테이블 트랜스포머 모델과 OCR 모듈을 활용하여 표와 이미지 처리에 강합니다. (https://www.docling.ai/)
- MarkitDown: 마이크로소프트의 오픈소스 경량 파서로, 유튜브 스크립트를 포함한 다양한 형태의 파일을 마크다운으로 변환합니다. (https://github.com/microsoft/markitdown)

In [40]:
# 2개의 파서 비교
sample_doc = docs[1]

for doc in docs:
    loader = PyMuPDFLoader(sample_doc, mode='single')
    docling_loader = DoclingLoader(sample_doc, export_type ='markdown')
    pymupdf_docs = loader.load()
    print('PyMuPDF 처리 완료! 총 글자수:', len(pymupdf_docs[0].page_content))
    docling_docs = docling_loader.load()
    print('Docling 처리 완료! 총 글자수:', len(docling_docs[0].page_content))

    break # 1개만 체크하고 종료


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


PyMuPDF 처리 완료! 총 글자수: 56725


[INFO] 2026-03-25 13:07:52,560 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-25 13:07:52,570 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.11/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-25 13:07:52,571 [RapidOCR] main.py:53: Using /usr/local/lib/python3.11/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-25 13:07:52,672 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-25 13:07:52,675 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.11/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-25 13:07:52,675 [RapidOCR] main.py:53: Using /usr/local/lib/python3.11/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-25 13:07:52,718 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-25 13:07:52,732 [RapidOCR] download_file.py:60: File exists and is valid: /usr/loc

Docling 처리 완료! 총 글자수: 118113


두 파서의 결과를 간단하게 비교해 보겠습니다.

In [41]:
print(pymupdf_docs[0].page_content[0:4000])

수신일반
인재개발팀 정지혜

1
1. 창구직원전결지침
가. 목적
- 이지침은영업점장에게위임된전결권일부를하위직원에게부여함으로써업무능률증진을기하고책임의
소재를명확히함을목적으로한다.
나. 수임자
- 책임자또는사무분장명령에의하여수임된사무를계속적으로수행하는창구직원(대리및행원)으로한다.
다. 위임의절차
- 영업점장은사무분장명령부에의하여전결권을위임하고동명령부에'창구직원전결지침에서정하는사항’
이라기재하여수임자의날인을받아야한다.
라. 전결권의한도및범위
1) 영업점장은[별표]창구직원전결기준표에해당하는사항을전조의절차에의하여위임함을원칙으로한다.
다만, 전결권위임이부적당하다고인정할경우에는전결권의일부또는전부를위임하지아니할수있다.
2) 영업점장은전항의위임기준에규정하지아니한사항이라하더라도위임기준과동등하다고판단되는사항에
대하여는자신의책임으로그권한의일부를위임할수있다.
3) 수임자는그전결한도내의사항이라도특히중요하다고인정되는것, 이례적이거나이의가있는것,
기타정규적인처리사항이아닌것에대하여는그상위직계자의결정또는지시를받아서처리하여야한다.
4) 복수결재를요하는업무처리시책임자가1인인경우는반드시영업점장을포함하여사후결재를하여야
한다.(단독결재금지)
5) 책임자로서창구업무를수행하는경우, 타책임자또는영업점장이결재를하여야한다.
마. 전결권행사의기준
1) 수임자가부재또는기타사유로전결권을행사할수없는경우에는직속상위책임자가그전결권을행사
한다.
2) 창구직원이전표에전결권을행사할경우에는'전결'란에날인한다.
3) 대내문서발송이필요한경우, 발신자명의는영업점장으로하고발신자명의옆에'대'라고표시한후
수임자의사인(私印)을찍는다.
바. 수임자의결재방법
- 수임자의결재방법은수임자난에'전결'표시를하고최종결재자란에날인한다.
사. 감독
- 영업점장은여신취급에대한전결권위임과관련하여월1회이상특명감사등을통하여위임여신의실행
내역과채권서류의일치여부를확인하여야한다.
아. 책임
1) 영업점장은지침에따라직무전결권을위임한때에는수임자의업무처리에관한감독책임을지며, 수임자는
자신이처리한업무처리결과에대하여책임을진다. 다만, [라.3)]에해당되는경우에는그러하지아니

In [42]:
print(docling_docs[0].page_content[0:4000])

/\_2635 /\_2660 /\_1104 /\_1994 /\_3185/\_1/\_2687 /\_2769 /\_3340/\_1

## 1. 창구직원 전결 지침

## 가 . 목적

- -이 지침은 영업점장에게 위임된 전결권 일부를 하위직원에게 부여함으로써 업무능률증진을 기하고 책임의 소재를 명확히 함을 목적으로 한다 .

## 나 . 수임자

- -책임자 또는 사무분장 명령에 의하여 수임된 사무를 계속적으로 수행하는 창구직원 ( 대리 및 행원 ) 으로 한다 .

## 다 . 위임의 절차

- -영업점장은 사무분장 명령부에 의하여 전결권을 위임하고 동 명령부에 ' 창구직원 전결 지침에서 정하는 사항 ' 이라 기재하여 수임자의 날인을 받아야 한다 .

## 라 . 전결권의 한도 및 범위

- 1) 영업점장은 [ 별표 ] 창구직원 전결 기준표에 해당하는 사항을 전조의 절차에 의하여 위임함을 원칙으로 한다 . 다만 , 전결권 위임이 부적당하다고 인정할 경우에는 전결권의 일부 또는 전부를 위임하지 아니할 수 있다 .
- 2) 영업점장은 전항의 위임기준에 규정하지 아니한 사항이라 하더라도 위임기준과 동등하다고 판단되는 사항에 대하여는 자신의 책임으로 그 권한의 일부를 위임할 수 있다 .
- 3) 수임자는 그 전결한도내의 사항이라도 특히 중요하다고 인정되는 것 , 이례적이거나 이의가 있는 것 , 기타 정규적인 처리사항이 아닌 것에 대하여는 그 상위직계자의 결정 또는 지시를 받아서 처리하여야 한다 .
- 4) 복수결재를 요하는 업무처리시 책임자가 1 인인 경우는 반드시 영업점장을 포함하여 사후결재를 하여야 한다 .( 단독결재 금지 )
- 5) 책임자로서 창구업무를 수행하는 경우 , 타 책임자 또는 영업점장이 결재를 하여야 한다 .

## 마 . 전결권행사의 기준

- 1) 수임자가 부재 또는 기타 사유로 전결권을 행사할 수 없는 경우에는 직속 상위 책임자가 그 전결권을 행사 한다 .
- 2) 창구직원이 전표에 전결권을 행사할 경우에는 ' 전결 ' 란에 날인한다 .
- 3) 

사전에 마크다운으로 추출된 문서를 폴더에서 가져오겠습니다.

In [ ]:
# Docling 처리 코드 (실행 X)
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

# markdown 폴더가 없으면 생성
os.makedirs('markdowns', exist_ok=True)

def save_markdown_from_doc(doc_path):
    loader = DoclingLoader(doc_path, export_type='markdown')
    # Lazy load: 문서를 하나씩 처리 (load()가 내부적으로 lazy하지 않다면 여기에서 조절 불가)
    docling_docs = loader.load()
    basename = os.path.basename(doc_path)
    md_filename = os.path.splitext(basename)[0] + ".md"
    md_path = os.path.join('markdowns', md_filename)
    with open(md_path, 'w', encoding='utf-8') as f:
        for d in docling_docs:
            f.write(d.page_content)
            f.write('\n\n')
    return md_path

with ThreadPoolExecutor() as executor:
    futures = [executor.submit(save_markdown_from_doc, doc_path) for doc_path in docs]
    for future in as_completed(futures):
        try:
            md_file = future.result()
            print(f"{md_file} 저장 완료")
        except Exception as e:
            print(f"오류 발생: {e}")


[INFO] 2026-03-25 13:11:21,865 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-25 13:11:21,872 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.11/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-25 13:11:21,873 [RapidOCR] main.py:53: Using /usr/local/lib/python3.11/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-25 13:11:21,972 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-25 13:11:21,975 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.11/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-25 13:11:21,977 [RapidOCR] main.py:53: Using /usr/local/lib/python3.11/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-25 13:11:22,020 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-25 13:11:22,038 [RapidOCR] download_file.py:60: File exists and is valid: /usr/loc

markdowns/금융실명거래 업무해설.md 저장 완료


[WARNING] 2026-03-25 13:14:41,641 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-03-25 13:14:41,902 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!


markdowns/수신일반.md 저장 완료


In [47]:
reports = glob('data/markdowns/*.md')
reports

['data/markdowns/여신일반.md',
 'data/markdowns/수신일반.md',
 'data/markdowns/금융실명거래 업무해설.md']

In [48]:
documents = []
for report in reports:
    loader = TextLoader(report)
    docs = loader.load()
    print('문서 로드 완료:', docs[0].metadata, docs[0].page_content[0:10], '...', len(docs[0].page_content))
    documents += docs
len(documents)


문서 로드 완료: {'source': 'data/markdowns/여신일반.md'} ## /\_2504 ... 254178
문서 로드 완료: {'source': 'data/markdowns/수신일반.md'} /\_2635 /\ ... 118115
문서 로드 완료: {'source': 'data/markdowns/금융실명거래 업무해설.md'} ## 금융실명거래  ... 253539


3

간단한 전처리를 수행합니다.

In [49]:
def preprocess(docs):
    import re
    def clean_text(doc):
        # 주요 이상 문자들만 제거: &와 ; 포함 HTML 특수문자 표현, 기타 혼란스러운 기호
        text = doc.page_content
        orig_len = len(text)

        # &amp, &lt 등 HTML 엔티티 기본 패턴 제거
        text1 = re.sub(r'&[a-zA-Z0-9#]+;', '', text)
        n1 = len(text) - len(text1)
        text = text1

        # 남아있는 이스케이프 문자(\xa0 등) 제거
        text2 = re.sub(r'[\u200b\u00a0]', '', text)
        n2 = len(text) - len(text2)
        text = text2

        # 탭/개행(\t, \n)은 보존. 단, 연속공백만 정리하되, \t와 \n은 공백으로 여기지 않는다. <- 표 있어서 처리안함
        # ex: '  foo\t\tbar\n\nbaz  ' → (공백 유지, 탭/엔터 유지, 맨 앞뒤strip만)
        # 연속공백(' ')만 정리 (이어서 \t, \n 전처리 영향 배제)
        # (공백 여러개 → 1개)
        text3 = re.sub(r' {2,}', ' ', text2)
        n3 = len(text2) - len(text3)
        text = text3.strip()
        n4 = len(text3) - len(text)

        removed = n1 + n2 + n3 + n4
        doc.page_content = text
        # 로그로 각 문서에서 제거된 문자의 개수를 표시
        print(f"clean_text: {removed} 문자를 제거함 (원본 {orig_len} → {len(text)})")
        return doc

    preprocessed_docs = []
    for doc in docs:
        doc = clean_text(doc)
        preprocessed_docs.append(doc)
    return preprocessed_docs

documents = preprocess(documents)

clean_text: 31297 문자를 제거함 (원본 254178 → 222881)
clean_text: 34710 문자를 제거함 (원본 118115 → 83405)
clean_text: 43561 문자를 제거함 (원본 253539 → 209978)


In [50]:
documents[0].metadata, documents[0].page_content[0:1000]

({'source': 'data/markdowns/여신일반.md'},
 "## /\\_2504 /\\_2344 /\\_2636 /\\_1992\n\n/\\_2635 /\\_2660 /\\_1104 /\\_1994 /\\_3185/\\_1/\\_1251 /\\_1155 /\\_1975 /\\_13/\\_1 /\\_2633 /\\_1245 /\\_2180\n\n## 1. 목적\n\n- -본 여신전결규정 ( 이하 ' 전결기준＂이라 함 ) 은 ' 개별차주 ' 를 기준으로 ' 개별차주 ' 에 대한 Exposure 운용에 관한 여신직무전결권 범위를 명확히 함을 목적으로 한다 .\n\n## 2. 용어\n\n- 'Exposure' 란 , ' 개벌차주 ' 의 채무 불이행시 은행에 손해를 끼칠 수 있는 모든 종류의 난내 , 난외 자산운용액의 한도기준 합계액을 말한다 . 단 , 본 전결기준에서는 개인회원에 대한 신용카드 한도는 제외한다 .\n- ' 담보한도 '( 또는 ' 담보여신 ') 라 함은 「담보 및 감정심사 지침」에서 정한 ' 유효담보가액 ' 이내에서 운용할 수 있는 한도를 말한다 .\n- -' 신용한도 '( 또는 ' 신용여신 ') 라 함은 개별차주의 총 Exposure 에서 개별여신의 ' 유효담보가액 ' 을 합산한 금액을 차감한 한도기준 신용부분 합산금액을 말한다 .\n\n## 3. 전결기준의 구분\n\n- -전결기준은 ' 일반여신 ' 과 ' 별도한도 운용 대상여신 ' 으로 구분하고 , 별도한도 운용 대상여신 이외의 여신은 일반 여신으로 보며 , ' 제 2 장 일반여신 전결권 운용기준 ' 과 ' 제 3 장 별도한도 운용 대상여신 전결권 운용기준 ' 은 구분 하여 각각 운용한다 .\n- -' 일반여신 전결권 운용기준 ' 은 개별차주의 신용평가모형 ( 참조 : 「신용평가모형 관리 지침」에 따라 ' 기업 Exposure 대상 여신 ' 은 기업모형 A( 특수금융 / 신용등급 예외부여 공공기관 , 금융기관 포함 ), 기업모형 B, 소규모법인 , 개인 사업자 , 비재무평가기업 ( 특

## 청킹(Chunking): 문서 분할하기


전체 문서 내용을 LLM에 전달하는 방법은      
Context의 길이가 늘어나 성능이 저하될 수 있고 비용/시간 문제가 발생합니다.   

이번 실습에서는 글자수를 기준으로 해당 데이터를 청킹하여 일부를 Context에 입력하는 RAG 세팅을 구현합니다.

In [51]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# 0~1000, 800~1800, 1600~2600, ...
# 구분자(엔터, 공백, 탭 등)을 기준으로 나누는 방식
chunks = text_splitter.split_documents(documents)
print(len(chunks))

758


## Embedding과 Vector DB

RAG의 시맨틱 검색(Semantic Search)에서는 주어진 질문에 대해, 적절한 청크를 검색합니다.   

이를 정량적으로 계산하기 위해, 텍스트를 벡터로 변환한 임베딩 벡터의 유사도를 구하는데요.    

이 때문에 LLM과 별개로 임베딩 모델이 필요합니다.   
임베딩 모델로 텍스트를 변환한 이후 결과를 벡터 DB에 저장하여 검색에 사용합니다.    
이번 실습에서는 두 개의 임베딩 모델을 사용합니다.

- OpenAI text-embedding-3-large
  - 2024.1 출시, 대표적 온라인 임베딩 모델
- Qwen 3 Embedding 0.6B
  - 2025.6 출시, 가장 작은 오픈 임베딩 모델

In [52]:
openai_embeddings = OpenAIEmbeddings(model='text-embedding-3-large', chunk_size=100)
                                                                    # 병렬 처리시 최대 100개씩 처리

허깅페이스에 게시된 공개 모델을 불러옵니다.   
오픈 임베딩 모델에서 중요한 파라미터는 다음과 같습니다.

- 파라미터 수 : 큰 임베딩 모델의 크기는 LLM에 육박합니다. GPU를 고려하여 선택합니다.
- Max Tokens: 임베딩 모델의 최대 토큰보다 큰 데이터를 입력하면, 앞부분만을 이용해 계산하게 되므로 적절한 검색이 되지 않을 수 있습니다.
- 임베딩 차원: 큰 차원의 벡터를 생성하는 임베딩 모델은 검색 속도가 감소합니다.

In [53]:
# HuggingFace 주소 (google/embeddinggemma-300m, Qwen/Qwen3-Embedding-0.6B, baai/bge-m3 등)

model_name = 'Qwen/Qwen3-Embedding-0.6B'
# 실제 주소: https://hf.co/Qwen/Qwen3-Embedding-0.6B

# CPU 설정으로 모델 불러오기
emb_model = SentenceTransformer(model_name, device='cpu',model_kwargs={'dtype':torch.bfloat16})

# 로컬 폴더에 모델 저장하기
emb_model.save('outputs/vectordb/embedding')

# 모델 메모리에서 삭제
del emb_model
gc.collect()

16286

In [54]:
# 허깅페이스 포맷의 임베딩 모델 불러오기
# tokenizer 오류는 무시 (수정예정)
qwen3_embeddings = HuggingFaceEmbeddings(model_name= 'outputs/vectordb/embedding',
                                   model_kwargs={'device':'cuda', 'model_kwargs':{'torch_dtype':torch.bfloat16}})
                                                    # gpu 사용하기                               bfloat16
# GPU 로드된 것 확인
print('임베딩 모델 로드 완료!')

`torch_dtype` is deprecated! Use `dtype` instead!
The tokenizer you are loading from 'outputs/vectordb/embedding' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


임베딩 모델 로드 완료!


### 참고) OllamaEmbeddings


In [ ]:
# # 'ollama pull qwen3-embedding:0.6b'로 모델 다운로드해야 함
# from langchain_ollama import OllamaEmbeddings

# ollama_embeddings = OllamaEmbeddings(model='qwen3-embedding:0.6b')
# 이후 동일하게 사용 가능!

## Vector DB에 저장하기   

벡터의 효율적인 검색을 지원하는 Vector DB에 저장합니다.   
이번 실습에서는 Qdrant(https://qdrant.tech/documentation/)를 사용합니다.
- **대부분의 벡터 DB는 동시성 처리와 안정적 서빙을 위한 Docker 세팅을 권장합니다. 추후에는 효율적인 세팅을 위해 도커를 사용해 주세요**

In [55]:
uuidstr = str(uuid.uuid4())[0:6]
# uuid: 랜덤 문자열(셀 중복 실행시 DB 중첩 방지)

client = QdrantClient(path=f"outputs/vectordb/qdrant_{uuidstr}")

client.create_collection(
    collection_name=f"AI_Reports",
    vectors_config=VectorParams(size=3072, distance=Distance.EUCLID),
    # EUCLID: 벡터간의 직선거리로 유사도 계산 <--> COSINE: 벡터간의 각도로 유사도 계산
)

True

In [56]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name=f"AI_Reports",
    embedding=openai_embeddings,
    distance=Distance.EUCLID,
)

vector_store.add_documents(chunks)
print("Vector Database에 청크 추가 완료!")

Vector Database에 청크 추가 완료!


RAG를 하기 전, 비교를 위해 LLM에게 질문해 보겠습니다.

In [57]:
# Test
for s in llm.stream("여권으로 실명확인이 가능한가요?"):
    print(s.content,end='')

😊

In South Korea, the answer is yes. The passport (여권) can be used to verify an individual's identity through a process called "real-name verification" (실명확인).

Real-name verification is a system that matches the holder's name with their biometric information, such as fingerprints or facial recognition data, stored in government databases. This helps prevent fraudulent activities and ensures that individuals are who they claim to be.

Here's how it works:

1. When applying for certain services or documents (such as opening a bank account, purchasing property, or applying for a job), you may need to provide your passport.
2. The relevant authorities will then verify the holder's identity by comparing the name on the passport with their biometric information stored in government databases.
3. If the verification is successful, it confirms that the individual is who they claim to be.

This system has been implemented to enhance national security and prevent fraudulent activities, such as

Knowledge Cutoff 이후의 내용에 대한 질문이므로, 정확하지 않으며 환각이 발생합니다.  

LLM에 벡터 DB 기반의 검색기를 연결하여 이를 해결해 보겠습니다.

db로부터 retriever를 구성합니다.

In [58]:
# Top 5 Search(기본값은 4)
retriever = vector_store.as_retriever(search_kwargs={'k':5})

# 총 컨텍스트는 몇글자일까?
# 5,000 --> 1,000 * 5

In [59]:
context = retriever.invoke("여권으로 실명확인이 가능한가요?")
context

[Document(metadata={'source': 'data/markdowns/금융실명거래 업무해설.md', '_id': '93301ec6611e4b95867c65d6cc065e55', '_collection_name': 'AI_Reports'}, page_content='| 명의인 | 신청인 | 실명확인에 필요한 서류 | 보관서류 |\n|----------|--------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------|\n| 기 | 보호시설에 있는 미성년자 | 본인의 주민등록등본 국가 또는 지방자치단체 운영 보호시설의 경우 ｢ 보호시설에 있는 미성년자의 후견 직무에 관한법률 ｣ 의 규정에 의하여 공설 보호시설의 장임을 확인할 수 있는 서류, 그 외의 보호시설의 경우 지방자치단체에서 미성년자의 성명과 주민등록번호를 확인한 서류 후견의직무를수행하는자의실명확인증표 불법·탈법 차명거래 금지 설명확인서 | 본인의 주민등록등본 원본, 후견인의 실명 확인증표 사본, 보호 시설의 장임을 확인 하는서류등, 불법· 탈법 차명거래 금지 설명확인서 원본 |\n| 

위 검색 결과를 전처리하여, LLM의 프롬프트로 넣기 위한 함수를 구성합니다.

In [60]:
# List[Document] --> String
# metadata의 source를 보존하여 추가
def format_docs(docs):
    return " \n\n---\n\n".join(
        [f"Source: {doc.metadata['source']}\nContent: {doc.page_content}"
                                for doc in docs])
    # join : 구분자를 기준으로 스트링 리스트를 하나의 스트링으로 연결

print(format_docs(context))

Source: data/markdowns/금융실명거래 업무해설.md
Content: | 명의인 | 신청인 | 실명확인에 필요한 서류 | 보관서류 |
|----------|--------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------|
| 기 | 보호시설에 있는 미성년자 | 본인의 주민등록등본 국가 또는 지방자치단체 운영 보호시설의 경우 ｢ 보호시설에 있는 미성년자의 후견 직무에 관한법률 ｣ 의 규정에 의하여 공설 보호시설의 장임을 확인할 수 있는 서류, 그 외의 보호시설의 경우 지방자치단체에서 미성년자의 성명과 주민등록번호를 확인한 서류 후견의직무를수행하는자의실명확인증표 불법·탈법 차명거래 금지 설명확인서 | 본인의 주민등록등본 원본, 후견인의 실명 확인증표 사본, 보호 시설의 장임을 확인 하는서류등, 불법· 탈법 차명거래 금지 설명확인서 원본 |
| 타 | 군인·경찰 | 소속부대장·경찰관서장이 성명과 주민 등록번호를 확인한 서류 ※ 부대장：행정권한이있는대대급단위 부대장을 말함 불법·탈법 차명거래 금지 설명확인서 | 실명확인서류 원본, 불법·탈법 

구성한 format_docs 함수는 이후 체인에 포함합니다.

## Prompt & Chain

In [61]:
prompt = ChatPromptTemplate([
    ("system", '''당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다양한 출처의 보고서 일부 내용이 Context로 주어집니다.
Context의 내용을 바탕으로 Question에 대한 답변을 제공하세요.
추가적인 정보를 위해 출처를 항상 포함하세요.

만약 Context가 질문과 무관하거나 관련 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.'''),
    ("human",'''Context: {context}
---
Question: {question}''')])
prompt.pretty_print()

================================ System Message ================================

당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다양한 출처의 보고서 일부 내용이 Context로 주어집니다.
Context의 내용을 바탕으로 Question에 대한 답변을 제공하세요.
추가적인 정보를 위해 출처를 항상 포함하세요.

만약 Context가 질문과 무관하거나 관련 정보가 없다면,
"정보가 부족하여 답변할 수 없습니다."를 출력하세요.

================================ Human Message =================================

Context: {context}
---
Question: {question}


## Chain

RAG를 수행하기 위한 Chain을 만듭니다.

In [62]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
    # Llm 출력을 문자열로 변환
)

# RAG Chain 흐름도:
# Question → ┬→ Retriever →  format_docs → context ┐
#            └→ RunnablePassthrough → question ────┤
#                                                  ↓
#                                               Prompt → LLM → Answer

In [63]:
result = rag_chain.invoke("여권으로 실명확인이 가능한가요?")
print(result)

Context:
여권, 재외국민용 주민등록증, 여권, 국내발급 운전면허증 불법·탈법 차명거래 금지 설명확인서 | 실명확인증표 사본, 불법·탈법 차명거래 금지설명확인서원本

Answer:
예시 ⑤에 따라, 재외국민용 주민등록증이 포함된 경우 여권으로 실명확인이 가능합니다. 

Translation: According to the example ⑤, yes, a passport can be used for real-name confirmation when accompanied by a resident registration certificate for foreign nationals.

Source: data/markdowns/금융실명거래 업무해설.md


In [66]:
result = rag_chain.invoke("30세 골드 VIP인 광은이가 창구에서 타행으로 500만원을 송금 시 송금수수료는 얼마인가?")
print(result)

Based on the provided context, I found relevant information that can help me answer your question.

In the "수신 일반" document, there is a table under the section "창구송금수수료 (타행송금)" that lists the fees for sending money to other banks. According to this table:

* For amounts over 10 million won but less than or equal to 100 million won, the fee is 3,000 won.

Since you are asking about sending 500 million won, which falls within this range, I would answer that the wire transfer fee for a 30-year-old Gold VIP customer like 광은이가 is 3,000 won.


In [68]:
result = rag_chain.invoke("한도거래여신 300백만원 기한연장 시 한도소진율이 27%(전전월말 이전 과거 6개월 평균고진율 기준)인 경우 영업점장전결로 기한연장 시 연장 가능금액은 최대 얼마인가?")
print(result)

Based on the context, I will try to provide an answer.

According to the content, when considering a limit transaction credit with a maturity limit of 300 million won and a utilization rate of 27% (based on the average utilization rate for the past 6 months prior to the previous month), we can apply the following rules:

1. If the utilization rate is 50% or higher, it is not subject to the limitation.
2. However, if the utilization rate is less than 50%, but greater than 30%, we need to apply the maximum amount of borrowing allowed based on the original loan amount.

According to the content, if the utilization rate is between 30% and 50%, we can calculate the maximum borrowing amount by applying the ratio of the unused amount (100% - utilization rate) to the original loan amount. In this case, the maximum borrowing amount would be:

Maximum borrowing amount = Original loan amount × (1 - Utilization rate)
= 300 million won × (1 - 0.27)
≈ 219.5 million won

So, in this scenario, the max

In [73]:
rag_chain.invoke("차주의 총 금융부채 상환부담을 판단하기 위하여 신청하는 차주의 연간 소득 대비 연간 금융부채 원리금 상환액 비율은 몇 퍼센트인가요?")

"Based on the provided context, the total debt service-to-income (DTI) ratio is calculated as follows:\n\n* For a single household: 40% or less\n* For multiple households (2 or more): 30% or less\n\nThe DTI ratio is used to determine the borrower's ability to repay their debts."

만약 Context가 포함된 RAG 결과를 보고 싶다면, RunnableParallel을 사용하면 됩니다.

In [74]:
rag_chain_from_docs = (
    prompt
    | llm
    | StrOutputParser()
)

rag_chain_with_source = RunnableParallel(
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)
# assign : 결과를 받아서 새로운 인수 추가하고 원래 결과와 함께 전달
# {context, question} + {answer}

rag_chain_with_source.invoke("자금용도가 주택구입목족이고, 주택보유수는 무주택세대, 지역구분은 기타지역이고, 생애최초주택구입 미대상인 조건의 주택담보대출 담보인정비율은 몇 퍼센트인가요?")
 
# retriever가 1번 실행됨
# retriever의 실행 결과를 rag_chain_from_docs 에 넘겨주기 때문에

{'context': "Source: data/markdowns/여신일반.md\nContent: ( 나 ) 주택을 보유하고 있는 세대에 대해 기타지역에서 주택을 추가 구입할 목적으로 해당 신규 주택을 담보로 대출을 취급하는 경우에는 제 (1) 호에서 정한 담보인정비율에 10%p 를차감하여야한다 . 다만 , 기존 주택의 주택매매계약을 체결하고 계약금을 받은 사실을 증명한 경우에는 차감하지 않을 수 있다 .\n\n( 라 ) 생애최초주택구매자의주택구입목적주택담보대출의경우에는담보인정비율을 80% 이내에서 취급할 수 있다 . 다만 , 이 경우 주택담보대출금액은 6 억원을초과할수없다 .\n\n## [ 참고 245-1] 주택담보대출에대한 담보인정비율\n\n| 자금 용도 | 주택보유수 ( 세대기준 ) | 지역구분 | 지역구분 | 지역구분 |\n|----------------|---------------------------|---------------------------|-----------------------|------------|\n| 자금 용도 | 주택보유수 ( 세대기준 ) | 투기지역 및 투기과 열지구 | 조정대상지역 | 기타지역 |\n| 주택 구입 목적 | 무주택 세대 | 50% 주 1) | 50% 주 1) | 70% |\n| | 1 주택 보유 세대 | 50% 주 2) / 30% 주 3) | 50% 주 2) / 30% 주 3) | 60% |\n| 자금 | 2 주택 이상 보유 세대 | 30% 주 3) | 30% 주 3) | 60% |\n| 생활 안정 자금 | 1 주택 보유 세대 | 50% | 50% | 70% |\n| 생활 안정 자금 | 2 주택 이상 보유 세대 | 40% | 40% | 60% |\n\n- 주 1) 서민 · 실수요자의규제지역소재주택구입목적대출경우담보인정비율에 20% 이내로가산할수있다 .\n- 주 2) 규제지역 내 주택을 구입하는 1 주택 보유 세대에 대해 주택담보대출 실행일로\n\n부터 2 년 이내에 기존 주택을 처분 약정하여 대출 취급시 LTV

이번에는 같은 질문을 Qwen Embedding으로 수행해 보겠습니다.

In [75]:
uuidstr = str(uuid.uuid4())[0:6]
# uuid: 랜덤 문자열(셀 중복 실행시 DB 중첩 방지)

client = QdrantClient(path=f"outputs/vectordb/qdrant_{uuidstr}")

client.create_collection(
    collection_name=f"AI_Reports",
    vectors_config=VectorParams(size=1024, distance=Distance.EUCLID),
    # Qwen 3 0.6b : 1024차원
)

True

In [76]:
qwen_vector_store = QdrantVectorStore(
    client=client,
    collection_name=f"AI_Reports",
    embedding=qwen3_embeddings,
    distance=Distance.EUCLID,
)

# 20개씩 추가 (GPU에 따라 늘려도 됨)
for i in tqdm(range(0, len(chunks), 20)):
    qwen_vector_store.add_documents(chunks[i:min(i+20, len(chunks))])


100%|██████████| 38/38 [00:39<00:00,  1.05s/it]


In [77]:
qwen_retriever = qwen_vector_store.as_retriever(search_kwargs={'k':5})

In [78]:
qwen_rag_chain = (
    {"context": qwen_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [79]:
questions = ["여권으로 실명확인이 가능한가요?",
"30세 골드 VIP인 광은이가 창구에서 타행으로 500만원을 송금 시 송금수수료는 얼마인가?",
"한도거래여신 300백만원 기한연장 시 한도소진율이 27%(전전월말 이전 과거 6개월 평균고진율 기준)인 경우 영업점장전결로 기한연장 시 연장 가능금액은 최대 얼마인가?",
"차주의 총 금융부채 상환부담을 판단하기 위하여 신청하는 차주의 연간 소득 대비 연간 금융부채 원리금 상환액 비율은 몇 퍼센트인가요?"]

In [80]:
# Retriever 비교
result_qwen = qwen_retriever.batch(questions)

for i in range(len(questions)):
    print(f'Question: {questions[i]}')
    for j in result_qwen[i]:
        print('   ',j.page_content[0:50].replace('\n', ''))


Question: 여권으로 실명확인이 가능한가요?
    - 개인 주민등록증이 원칙. 다만, 국가기관 또는 지방자치단체, 유아교육법· 초중등교육법·
    | 명의인 | 신청인 | 실명확인에 필요한 서류 | 보관서류 ||-------------
    입출금이 자유로운예금(신탁), 적립식·거치식예금(신탁), 환매조건부채권· 국공채(통장거래)
    | 외국 국적동포 | 대리인 (영사 확인 또는 공증을 받은 경우) | 본인의 실명확인증표(
    금융실명거래 업무해설\_ 35## 비대면 실명거래 방법## 1. 비대면 실명확인 개
Question: 30세 골드 VIP인 광은이가 창구에서 타행으로 500만원을 송금 시 송금수수료는 얼마인가?
    | | | | 고 객 등 급 기 준 | 고 객 등 급 기 준 | 고 객 등 급 기 준 | 
    ## 다 . 우대서비스 내용| 구 분 | 법인 등급 | 법인 등급 | 법인 등급 | 법
    수수료(원5,0005,0002,0002,0001,000-3
    ## 자 . 무통장 해지수수료- -요구성 예금의 무통장해지시 계좌단위로 받으며 , 해지
    ## 早今[VIP]型[H日I## 4. VIP( 법인고객 ) 우대서비스## 가 . 선
Question: 한도거래여신 300백만원 기한연장 시 한도소진율이 27%(전전월말 이전 과거 6개월 평균고진율 기준)인 경우 영업점장전결로 기한연장 시 연장 가능금액은 최대 얼마인가?
    ## 4. 전결권 운용기준- 가 . 기한연장 제한을 적용받지 않는 제외대상 여신인 경우
    주 ) 담보비율: 기한연장 시점의 재평가 금액 기준임- ■ 한도거래여신 기한연장 전
    - ① 기한연장 전결권 적용은 다음에서 정한 바에 따라 운용하며 , 최고 전결권자는 주관부
    - -「 5. 전결권의 제한」 앞면 이어서 -- 라 . 가계자금대출 , 서민금융관련대출 
    3. 한국거래소상장기업에대한대출4. 이자보상배율이 3 년연속동업계평균이상인기업에대한대출
Question: 차주의 총 금융부채 

In [81]:
result_openai = retriever.batch(questions)
result_openai

for i in range(len(questions)):
    print(f'Question: {questions[i]}')
    for j in result_openai[i]:
        print('   ',j.page_content[0:50].replace('\n', ''))


Question: 여권으로 실명확인이 가능한가요?
    | 명의인 | 신청인 | 실명확인에 필요한 서류 | 보관서류 ||----------|--
    ## 가. 개 인(주민등록증 발급대상자)Ⅱ 유형 별 대면 실명거래 방법
    | | 본 인 | 재외국민용 주민등록증, 재외국민 국 내거소신고증('16.6.30.까지 유
    | 명의인 | 신청인 | 실명확인에 필요한 서류 | 보관서류 ||-------------
    - 개인 주민등록증이 원칙. 다만, 국가기관 또는 지방자치단체, 유아교육법· 초중등교육법·
Question: 30세 골드 VIP인 광은이가 창구에서 타행으로 500만원을 송금 시 송금수수료는 얼마인가?
    수수료(원5,0005,0002,0002,0001,000-3
    | | | | 고 객 등 급 기 준 | 고 객 등 급 기 준 | 고 객 등 급 기 준 | 
    ## 자 . 무통장 해지수수료- -요구성 예금의 무통장해지시 계좌단위로 받으며 , 해지
    | 구 분 | 단 위 | 수수료 ( 원 ) | 비 고 ||-----------------
    | 100만원 이 하 | 본 인 대리인 | 실명확인 생략가능 | 실명확인 생략가능 || 
Question: 한도거래여신 300백만원 기한연장 시 한도소진율이 27%(전전월말 이전 과거 6개월 평균고진율 기준)인 경우 영업점장전결로 기한연장 시 연장 가능금액은 최대 얼마인가?
    주 ) 담보비율: 기한연장 시점의 재평가 금액 기준임- ■ 한도거래여신 기한연장 전
    - ③ 제 1 항내지제 2 항의연장건에 대해 기한연장 기준에 따라 영업점장이 전결로 연장코
    ## 4. 전결권 운용기준- 가 . 기한연장 제한을 적용받지 않는 제외대상 여신인 경우
    | 담보비율 주 ) 70% 미만 | 조건 ( 부 ) 2 적용대상여신 | ▷ 다음 각 호 중
    ## ( 사례 )담보여신 전결한도가 8 억원인 영업점장이 일반여신 8 억원을 보유하고 
Question: 차주의 총 금융부채 상환부

In [82]:
# 최종 결과 비교
answers = qwen_rag_chain.batch(questions)
for i in range(len(questions)):
    print(f'Question: {questions[i]}')
    print('   ',answers[i])
    print('========\n'*2)

Question: 여권으로 실명확인이 가능한가요?
    According to the context, yes, a passport can be used for real-name confirmation.

In the example of "외국인 (foreigner)" in the table, it is listed that a passport ("여권") can be used as a valid document for real-name confirmation.

Additionally, in the section ""비대면 실명확인을 적용할 수 있는 금융거래의 범위"" (Scope of non-face-to-face real-name confirmation), it is mentioned that "여권" is one of the recognized documents for real-name confirmation.

Question: 30세 골드 VIP인 광은이가 창구에서 타행으로 500만원을 송금 시 송금수수료는 얼마인가?
    Based on the provided context, for a 30-year-old Gold VIP customer who is sending 500 million KRW to another bank through a branch, the remittance fee would be:

* 창구송금수수료 (다른 은행으로 보낼 때) : 2,000 KRW (according to the "송금 및 기타수수료" table)

Note: The customer is considered a Gold VIP, which has a different rate compared to regular customers.

Question: 한도거래여신 300백만원 기한연장 시 한도소진율이 27%(전전월말 이전 과거 6개월 평균고진율 기준)인 경우 영업점장전결로 기한연장 시 연장 가능금액은 최대 얼마인가?
    Context: The conten

In [83]:
answers = rag_chain.batch(questions)
for i in range(len(questions)):
    print(f'Question: {questions[i]}')
    print('   ',answers[i])
    print('========\n'*2)

Question: 여권으로 실명확인이 가능한가요?
    According to the context provided, it seems that a passport (여권) can be used for real-name confirmation in certain cases.

For example, under "Ⅱ. 유형 별 대면 실명거래 방법" in the source material, it is mentioned that a person who has undergone an identity check by the relevant authority or has received a proof of identity from a government agency or a school (e.g., a student ID with a photo) can use their passport as a real-name confirmation document.

Additionally, under "## 실명확인증표의 예시" in the source material, it is listed as one of the possible real-name confirmation documents for foreigners, alongside other identity documents such as foreign registration certificates and driver's licenses.

Therefore, based on this information, it appears that a passport can be used for real-name confirmation in certain circumstances.

Question: 30세 골드 VIP인 광은이가 창구에서 타행으로 500만원을 송금 시 송금수수료는 얼마인가?
    Based on the context provided, I found a relevant table that shows the fees f